# 2. Connection

Construct an owner without I/O, then explicitly verify one exact device identity.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. Inspect a passive owner

`ShutterHub` is an application owner; its name does not imply a KEH3
connection. Construction and `connection_snapshot()` do not contact hardware.


In [ ]:
hub = ShutterHub(configs)
try:
    pprint(asdict(hub.connection_snapshot(TARGET)))
finally:
    hub.close()


## 2. Connect and verify identity

Enable `RUN_HARDWARE` only for an authorized read-only connection with
local config. D2XX needs the optional Python extra and native FTDI library;
VCP needs an exact COM port and is read-only commissioning. Neither route
has physical qualification in this project. Connect checks model, serial,
firmware and channel count. It does not change operating mode or output.


In [ ]:
if RUN_HARDWARE:
    hub = ShutterHub(configs)
    try:
        pprint(asdict(hub.connect(TARGET)))
        pprint(asdict(hub.identify_device(TARGET)))
    finally:
        hub.close()
else:
    print('Connection skipped; no device opened.')


Expected live result: `identity_verified=True`, the exact requested serial,
model `KSC101` and allowed firmware. On mismatch, ownership conflict or
timeout, stop and investigate the exact route; do not retry on another device.
`hub.close()` releases communication only. It does not send OFF or prove
optical closure. Only one notebook/CLI/GUI may own the controller at a time.
